In [1]:
import pandas as pd
import numpy as np

In [5]:
first_chunk = True

for chunk in pd.read_csv("../../../data/raw/PPIs/human_annotated_PPIs.txt", sep="\t",chunksize=50000, usecols=["symbol1", "symbol2", "lung"]):
    colomns_to_keep = chunk[['symbol1','symbol2','lung']]
    
    filtered = colomns_to_keep[colomns_to_keep['lung'] >= 1]
    
    if first_chunk:
        filtered.to_csv('../../../data/lung/lung_filter.csv', index=False, mode='w', header=True)
        first_chunk = False
    else:
        filtered.to_csv('../../../data/lung/lung_filter.csv', index=False, mode='a', header=False)

In [2]:
lung_filter = pd.read_csv('../../../data/lung/lung_filter.csv')
lung_filter.rename(columns={'symbol1':'A','symbol2':'B','kidney':'Combined_score'})

,A,B,lung
0,MAP2K4,FLNC,2
1,FNTA,ACVR1,2
2,GATA2,PML,2
3,RPA2,STAT3,2
4,ARF1,GGA3,2
...,...,...,...
1559929,LAMA2,FBLN2,2
1559930,BMP1,DSPP,1
1559931,SNAI1,LOXL1,2
1559932,FBLN1,COL18A1,2


In [3]:
lung_filter.isnull().any()

symbol1    False
symbol2    False
lung       False
dtype: bool

In [4]:
lung_filter[lung_filter['symbol1'] == lung_filter['symbol2']].sum()

symbol1    MCM2CSRP1KMT2APKD2LMO2RPLP2SNRPACLUFASNS100PSY...
symbol2    MCM2CSRP1KMT2APKD2LMO2RPLP2SNRPACLUFASNS100PSY...
lung                                                   11043
dtype: object

In [5]:
lung_filter = lung_filter[lung_filter['symbol1'] != lung_filter['symbol2']]

In [6]:
lung_filter.shape

(1554404, 3)

In [7]:
lung_filter.duplicated().sum()

np.int64(1074)

In [8]:
lung_filter = lung_filter.drop_duplicates()

In [9]:
lung_filter.shape

(1553330, 3)

In [10]:
pair = lung_filter[['symbol1','symbol2']]
print (pair)

        symbol1  symbol2
0        MAP2K4     FLNC
1          FNTA    ACVR1
2         GATA2      PML
3          RPA2    STAT3
4          ARF1     GGA3
...         ...      ...
1559929   LAMA2    FBLN2
1559930    BMP1     DSPP
1559931   SNAI1    LOXL1
1559932   FBLN1  COL18A1
1559933   FBLN1    LAMA2

[1553330 rows x 2 columns]


In [11]:
pair_filtered = pair.loc[pd.DataFrame(np.sort(pair.values, axis=1), index=pair.index).drop_duplicates().index]
print(pair_filtered)


        symbol1  symbol2
0        MAP2K4     FLNC
1          FNTA    ACVR1
2         GATA2      PML
3          RPA2    STAT3
4          ARF1     GGA3
...         ...      ...
1559929   LAMA2    FBLN2
1559930    BMP1     DSPP
1559931   SNAI1    LOXL1
1559932   FBLN1  COL18A1
1559933   FBLN1    LAMA2

[1548048 rows x 2 columns]


In [12]:
lung_filter = lung_filter.loc[pair_filtered.index]


In [13]:
lung_filter.shape


(1548048, 3)

In [14]:
expressions= pd.read_csv('../../../data/raw/depmap/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv', nrows=0)

In [15]:
expressions

,Unnamed: 0,SequencingID,ModelID,IsDefaultEntryForModel,ModelConditionID,IsDefaultEntryForMC,TSPAN6 (7105),TNMD (64102),DPM1 (8813),SCYL3 (57147),...,ATXN8 (724066),SMIM42 (117981789),NPBWR1 (2831),ACTL10 (170487),RNF228 (122319436),PANO1 (101927423),HRURF (120766137),PRRC2B (84726),F8A2 (474383),F8A1 (8263)


In [16]:
gene_cols = expressions.columns[6:]   
gene_cols

Index(['TSPAN6 (7105)', 'TNMD (64102)', 'DPM1 (8813)', 'SCYL3 (57147)',
       'FIRRM (55732)', 'FGR (2268)', 'CFH (3075)', 'FUCA2 (2519)',
       'GCLC (2729)', 'NFYA (4800)',
       ...
       'ATXN8 (724066)', 'SMIM42 (117981789)', 'NPBWR1 (2831)',
       'ACTL10 (170487)', 'RNF228 (122319436)', 'PANO1 (101927423)',
       'HRURF (120766137)', 'PRRC2B (84726)', 'F8A2 (474383)', 'F8A1 (8263)'],
      dtype='str', length=19215)

In [17]:
exprn_genes = expressions.columns[6:].str.split(" (",regex=False).str[0]

In [18]:
(~lung_filter["symbol1"].isin(exprn_genes)).sum()


np.int64(10441)

In [19]:
lung_filter[~lung_filter["symbol1"].isin(exprn_genes)]["symbol1"].nunique()

1051

In [20]:
(~lung_filter["symbol2"].isin(exprn_genes)).sum()

np.int64(10355)

In [21]:
lung_filter[~lung_filter["symbol2"].isin(exprn_genes)]["symbol2"].nunique()

736

In [22]:
lung_expr_filtered = lung_filter[lung_filter["symbol1"].isin(exprn_genes) & lung_filter["symbol2"].isin(exprn_genes)]

lung_expr_filtered.shape

(1527580, 3)

In [23]:
pd.concat([lung_expr_filtered["symbol1"], lung_expr_filtered["symbol2"]]).nunique()


18342

In [24]:
gene = pd.read_csv('../../../data/raw/depmap/CRISPRGeneEffect.csv',nrows=0)

In [25]:
gene_lab = gene.columns[1:].str.split(" (",regex=False).str[0]
gene_lab

Index(['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A3GALT2', 'A4GALT', 'A4GNT', 'AAAS',
       'AACS', 'AADAC',
       ...
       'ZWILCH', 'ZWINT', 'ZXDA', 'ZXDB', 'ZXDC', 'ZYG11A', 'ZYG11B', 'ZYX',
       'ZZEF1', 'ZZZ3'],
      dtype='str', length=18435)

In [26]:
ppi_genes = pd.concat([lung_expr_filtered["symbol1"], lung_expr_filtered["symbol2"]]).unique()

sum(pd.Series(ppi_genes).isin(gene_lab))

17849

In [27]:
lung_expr_filtered.rename(columns={'symbol1':'A','symbol2':'B','lung':'combined_score'})

,A,B,combined_score
0,MAP2K4,FLNC,2
1,FNTA,ACVR1,2
2,GATA2,PML,2
3,RPA2,STAT3,2
4,ARF1,GGA3,2
...,...,...,...
1559929,LAMA2,FBLN2,2
1559930,BMP1,DSPP,1
1559931,SNAI1,LOXL1,2
1559932,FBLN1,COL18A1,2


In [28]:
for i in range(0,len(lung_expr_filtered),100000):
    chunk = lung_expr_filtered.iloc[i:i+100000]
    if i == 0:
        chunk.to_csv('../../../data/lung/lung_PPI_final.csv',index=False,mode='w')
    else:
        chunk.to_csv('../../../data/lung/lung_PPI_final.csv',index=False,mode='a',header=False)

In [30]:
PPI_lung = pd.read_csv('../../../data/lung/lung_PPI_final.csv')

PPI_lung.rename(columns={'symbol1':'A','symbol2':'B','lung':'combined_score'}).to_csv('../../../data/lung/lung_PPI_final.csv', index=False)